# fuck you bitch ass modafockah

In [27]:
#importing libraries
import pandas as pd

In [28]:
base_path = '~/Repos/gabrielcruzg3/college/college/project/'
path_CSV = f'{base_path}MICRODADOS_ED_SUP_IES_2023.CSV'

# Etapa 1: Carregar o conjunto de dados
# Substitua 'caminho_para_o_arquivo.csv' pelo caminho do arquivo INEP
data = pd.read_csv(path_CSV, nrows=1, sep=";", encoding='latin-1')

# data_info = df.info()
data_head = data.head()

# data_info
data_head


,NU_ANO_CENSO,NO_REGIAO_IES,CO_REGIAO_IES,NO_UF_IES,SG_UF_IES,CO_UF_IES,NO_MUNICIPIO_IES,CO_MUNICIPIO_IES,IN_CAPITAL_IES,NO_MESORREGIAO_IES,...,QT_DOC_EX_60_MAIS,QT_DOC_EX_BRANCA,QT_DOC_EX_PRETA,QT_DOC_EX_PARDA,QT_DOC_EX_AMARELA,QT_DOC_EX_INDIGENA,QT_DOC_EX_COR_ND,QT_DOC_EX_BRA,QT_DOC_EX_EST,QT_DOC_EX_COM_DEFICIENCIA
0,2023,Centro-Oeste,5,Mato Grosso,MT,51,Cuiabá,5103403,1,Centro-Sul Mato-grossense,...,175,1003,85,454,38,6,2,1559,29,12


In [29]:
# Selecionando colunas úteis para começar a limpeza e análise inicial
# Vamos incluir uma variedade de colunas categóricas e numéricas para análise inicial.

selected_columns = [
    'NU_ANO_CENSO', 'NO_REGIAO_IES', 'CO_REGIAO_IES', 'NO_UF_IES', 'SG_UF_IES', 
    'CO_UF_IES', 'NO_MUNICIPIO_IES', 'CO_MUNICIPIO_IES', 'IN_CAPITAL_IES',
    'QT_DOC_EX_60_MAIS', 'QT_DOC_EX_BRANCA', 'QT_DOC_EX_PRETA', 'QT_DOC_EX_PARDA',
    'QT_DOC_EX_AMARELA', 'QT_DOC_EX_INDIGENA', 'QT_DOC_EX_COR_ND', 
    'QT_DOC_EX_BRA', 'QT_DOC_EX_EST', 'QT_DOC_EX_COM_DEFICIENCIA'
]

# Criando um subconjunto para focar na limpeza e análise
subset_df = data[selected_columns]

# Verificando valores ausentes e possíveis problemas de consistência
missing_values = subset_df.isnull().sum()
unique_values = subset_df.nunique()  # Contagem de valores únicos em cada coluna

missing_values, unique_values

(NU_ANO_CENSO                 0
 NO_REGIAO_IES                0
 CO_REGIAO_IES                0
 NO_UF_IES                    0
 SG_UF_IES                    0
 CO_UF_IES                    0
 NO_MUNICIPIO_IES             0
 CO_MUNICIPIO_IES             0
 IN_CAPITAL_IES               0
 QT_DOC_EX_60_MAIS            0
 QT_DOC_EX_BRANCA             0
 QT_DOC_EX_PRETA              0
 QT_DOC_EX_PARDA              0
 QT_DOC_EX_AMARELA            0
 QT_DOC_EX_INDIGENA           0
 QT_DOC_EX_COR_ND             0
 QT_DOC_EX_BRA                0
 QT_DOC_EX_EST                0
 QT_DOC_EX_COM_DEFICIENCIA    0
 dtype: int64,
 NU_ANO_CENSO                 1
 NO_REGIAO_IES                1
 CO_REGIAO_IES                1
 NO_UF_IES                    1
 SG_UF_IES                    1
 CO_UF_IES                    1
 NO_MUNICIPIO_IES             1
 CO_MUNICIPIO_IES             1
 IN_CAPITAL_IES               1
 QT_DOC_EX_60_MAIS            1
 QT_DOC_EX_BRANCA             1
 QT_DOC_EX_PRETA         

In [35]:
# 1. Verificar redundância em IDs de município
# Combinar os nomes e IDs de municípios e encontrar inconsistências
municipio_check = data[['NO_MUNICIPIO_IES', 'CO_MUNICIPIO_IES']].drop_duplicates()
municipio_redundancies = municipio_check[municipio_check.duplicated(subset=['CO_MUNICIPIO_IES'], keep=False)]

# 2. Normalizar variáveis categóricas
# Remover espaços extras e ajustar maiúsculas/minúsculas
data['NO_REGIAO_IES'] = data['NO_REGIAO_IES'].str.strip().str.title()
data['NO_UF_IES'] = data['NO_UF_IES'].str.strip().str.title()
data['NO_MUNICIPIO_IES'] = data['NO_MUNICIPIO_IES'].str.strip().str.title()

# 3. Análise estatística inicial
# Resumo estatístico das colunas numéricas
numeric_cols = data.select_dtypes(include=['int64', 'float64']).columns
stats_summary = data[numeric_cols].describe()

# Contagem de valores únicos para variáveis categóricas
categorical_cols = ['NO_REGIAO_IES', 'NO_UF_IES', 'NO_MUNICIPIO_IES', 'IN_CAPITAL_IES']
unique_values = {col: data[col].nunique() for col in categorical_cols}

# 4. Checar correlações
# Calcular correlações para variáveis numéricas
correlations = data[numeric_cols].corr()

# Exemplo de cruzamento geográfico com características docentes
docentes_por_regiao = data.groupby('NO_REGIAO_IES')['QT_DOC_EX_60_MAIS'].sum()

# Resultados
results = {
    "Municipio_Redundancies": municipio_redundancies,
    "Stats_Summary": stats_summary,
    "Unique_Values": unique_values,
    "Correlations": correlations,
    "Docentes_Por_Regiao": docentes_por_regiao
}

# Salvar resultados em arquivos CSV
export_path = f'{base_path}/analysis_export/'

municipio_redundancies.to_csv(export_path+'/municipio_redundancies.csv', index=False)
stats_summary.to_csv(export_path+'/stats_summary.csv')
pd.DataFrame.from_dict(unique_values, orient='index', columns=['Unique_Count']).to_csv(export_path+'/unique_values.csv')
correlations.to_csv(export_path+'/correlations.csv')
docentes_por_regiao.to_csv(export_path+'/docentes_por_regiao.csv')


print("Análise concluída. Resultados salvos como arquivos CSV.\n")
print("municipio_redundancies.csv: Inconsistências entre nomes e IDs de municípios.")
print("stats_summary.csv: Estatísticas descritivas das colunas numéricas.")
print("unique_values.csv: Contagem de valores únicos em variáveis categóricas.")
print("correlations.csv: Matriz de correlação entre variáveis numéricas.")
print("docentes_por_regiao.csv: Total de docentes com mais de 60 anos por região.")

Análise concluída. Resultados salvos como arquivos CSV.

municipio_redundancies.csv: Inconsistências entre nomes e IDs de municípios.
stats_summary.csv: Estatísticas descritivas das colunas numéricas.
unique_values.csv: Contagem de valores únicos em variáveis categóricas.
correlations.csv: Matriz de correlação entre variáveis numéricas.
docentes_por_regiao.csv: Total de docentes com mais de 60 anos por região.


In [ ]:
# Importing necessary libraries
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# 1. Data Cleaning (handling missing values, duplicates, etc.)
# Drop duplicate rows
data.drop_duplicates(inplace=True)

# Fill missing values (for numerical columns, using the median or mean is common)
data.fillna(data.median(), inplace=True)

# If you need to remove columns with a lot of missing values
# data.dropna(axis=1, thresh=0.5*len(data), inplace=True)

# 2. Statistical Summary
# Get basic descriptive statistics
summary = data.describe()

# Check for missing values
missing_values = data.isnull().sum()

# 3. Correlation Analysis
# Calculate the correlation matrix
correlation_matrix = data.corr()

# Display correlation of a specific column, for example 'QT_MAT_ATIV_EXTRACURRICULAR' with others
corr_column = data[['QT_MAT_ATIV_EXTRACURRICULAR', 'QT_CONC_ATIV_EXTRACURRICULAR', 'QT_MOB_ACADEMICA']].corr()

# 4. Feature Scaling (Standardizing data for clustering and PCA)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data.select_dtypes(include=[np.number]))

# 5. Dimensionality Reduction using PCA
pca = PCA(n_components=2)  # Reduce to 2 components for easier analysis
pca_components = pca.fit_transform(scaled_data)

# Add PCA results to the dataframe
data['PCA1'] = pca_components[:, 0]
data['PCA2'] = pca_components[:, 1]

# 6. Clustering Analysis (KMeans Clustering)
# Number of clusters chosen arbitrarily here (can use the elbow method to determine optimal clusters)
kmeans = KMeans(n_clusters=3, random_state=42)
data['Cluster'] = kmeans.fit_predict(scaled_data)

# 7. Insights: Show the top rows of the modified data
data.head()

# 8. Summary Results
print("Statistical Summary:")
print(summary)

print("\nMissing Values by Column:")
print(missing_values)

print("\nCorrelation of Specific Columns:")
print(corr_column)

print("\nData with PCA components and clusters:")
print(data[['PCA1', 'PCA2', 'Cluster']].head())



KeyError: "None of [Index(['QT_MAT_ATIV_EXTRACURRICULAR', 'QT_CONC_ATIV_EXTRACURRICULAR',\n       'QT_MOB_ACADEMICA'],\n      dtype='object')] are in the [columns]"